# Studi Kasus: Tim Penggali Sumur (Data Mining vs Machine Learning)

**Latar Belakang:**
Dua tim penggali sumur memiliki pendekatan berbeda dalam menemukan sumber mata air di wilayah baru.
* **Tim DM (Pendekatan Heuristik/Intuitif):** Langsung menggali berdasarkan intuisi atau pengamatan sekilas di lokasi (aturan statis/tebakan). Mereka tidak memanfaatkan data masa lalu secara maksimal.
* **Tim ML (Pendekatan Prediktif):** Mengumpulkan data historis pengeboran sebelumnya (kepadatan tanah, topografi, gravitasi). Mereka "belajar" dari kegagalan dan keberhasilan masa lalu untuk memprediksi probabilitas keberhasilan di wilayah baru sebelum mulai menggali.

**Tujuan Notebook:**
Mensimulasikan perbedaan hasil kerja (akurasi penemuan air) antara tebakan intuisi manusia (Tim DM) dan model prediktif yang dilatih dari pengalaman (Tim ML).

## 1. Persiapan Data Historis dan Data Wilayah Baru
Kita akan membuat data *dummy*. Pola rahasia alam yang menentukan adanya air adalah interaksi yang cukup kompleks antara kepadatan tanah, kemiringan topografi, dan anomali gravitasi.

In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Fungsi untuk membuat data simulasi wilayah pengeboran
def buat_data_pengeboran(jumlah_titik):
    np.random.seed(42)
    kepadatan_tanah = np.random.uniform(1, 10, jumlah_titik)
    kemiringan_topografi = np.random.uniform(0, 45, jumlah_titik)
    anomali_gravitasi = np.random.uniform(0, 1, jumlah_titik)
    
    # Pola rahasia alam (Hukum Fisika fiktif untuk simulasi ini)
    # Air ditemukan JIKA (kepadatan sedang DAN tanah datar) ATAU (anomali gravitasi tinggi)
    sumber_air = []
    for padat, miring, gravitasi in zip(kepadatan_tanah, kemiringan_topografi, anomali_gravitasi):
        if ((3 < padat < 7 and miring < 15) or (gravitasi > 0.8)) and np.random.rand() > 0.1:
            sumber_air.append(1) # Sukses nemu air
        else:
            sumber_air.append(0) # Gagal / Kering
            
    return pd.DataFrame({
        'kepadatan_tanah': kepadatan_tanah,
        'kemiringan_topografi': kemiringan_topografi,
        'anomali_gravitasi': anomali_gravitasi,
        'sumber_air': sumber_air
    })

# 1. Data Historis (Pengalaman masa lalu di wilayah lain) - 1000 titik
data_pengalaman = buat_data_pengeboran(1000)

# 2. Data Wilayah Baru (Lokasi yang akan dieksekusi hari ini) - 200 titik baru
data_wilayah_baru = buat_data_pengeboran(200)

# Pisahkan fitur dan target untuk wilayah baru (untuk pengecekan nanti)
X_baru = data_wilayah_baru[['kepadatan_tanah', 'kemiringan_topografi', 'anomali_gravitasi']]
y_asli_wilayah_baru = data_wilayah_baru['sumber_air']

print("Data Historis dan Data Wilayah Baru berhasil disiapkan!")
print(f"Proporsi sumur yang ada airnya di masa lalu: {data_pengalaman['sumber_air'].mean()*100:.1f}%")

Data Historis dan Data Wilayah Baru berhasil disiapkan!
Proporsi sumur yang ada airnya di masa lalu: 29.3%


## 2. Eksekusi Tim DM (Mengandalkan Intuisi & Aturan Statis)

Tim DM sampai di lokasi baru. Berdasarkan intuisi manusia secara umum, mereka beranggapan: *"Biasanya air itu ada di dataran yang landai/datar. Ayo kita bor semua titik yang kemiringannya di bawah 15 derajat!"*

Mereka **tidak** melatih model apapun dari `data_pengalaman`. Mereka langsung mengeksekusi aturan logika mereka ke `X_baru`.

In [7]:
# Tim DM membuat aturan statis berdasarkan intuisi
def prediksi_tim_dm(data_lokasi):
    prediksi = []
    for kemiringan in data_lokasi['kemiringan_topografi']:
        if kemiringan < 15: # Aturan intuisi manusia
            prediksi.append(1) # Bor di sini
        else:
            prediksi.append(0) # Jangan bor
    return prediksi

# Tim DM melakukan pengeboran di wilayah baru
hasil_tebakan_dm = prediksi_tim_dm(X_baru)

# Evaluasi tingkat keberhasilan Tim DM
akurasi_dm = accuracy_score(y_asli_wilayah_baru, hasil_tebakan_dm)
print(f"Akurasi Keberhasilan Tim DM (Intuisi/Rules): {akurasi_dm * 100:.1f}%")

Akurasi Keberhasilan Tim DM (Intuisi/Rules): 73.0%


## 3. Eksekusi Tim ML (Belajar dari Pengalaman)
Sebelum berangkat ke lokasi baru, Tim ML membuka catatan `data_pengalaman`. Mereka membiarkan algoritme **Random Forest** mempelajari pola tersembunyi dari interaksi kepadatan tanah, topografi, dan gravitasi terhadap temuan air di masa lalu. 

Setelah mesin menjadi pintar, barulah mereka memprediksi lokasi di `X_baru`.

In [8]:
# 1. Persiapkan data pembelajaran (Pengalaman Masa Lalu)
X_pengalaman = data_pengalaman[['kepadatan_tanah', 'kemiringan_topografi', 'anomali_gravitasi']]
y_pengalaman = data_pengalaman['sumber_air']

# 2. Mesin Belajar (Training Phase)
model_ml = RandomForestClassifier(n_estimators=100, random_state=42)
model_ml.fit(X_pengalaman, y_pengalaman)
print("Tim ML selesai melatih model dari pengalaman masa lalu.")

# 3. Mesin Memprediksi Wilayah Baru (Prediction Phase)
hasil_prediksi_ml = model_ml.predict(X_baru)

# Evaluasi tingkat keberhasilan Tim ML
akurasi_ml = accuracy_score(y_asli_wilayah_baru, hasil_prediksi_ml)
print(f"Akurasi Keberhasilan Tim ML (Pembelajaran Mesin): {akurasi_ml * 100:.1f}%")

Tim ML selesai melatih model dari pengalaman masa lalu.
Akurasi Keberhasilan Tim ML (Pembelajaran Mesin): 96.0%


## 4. Kesimpulan Perbandingan
Di bawah ini kita melihat bagaimana pengalaman mengalahkan sekadar intuisi.

In [9]:
df_komparasi = pd.DataFrame({
    'Tim': ['Tim DM (Intuisi/Rules)', 'Tim ML (Predictive Model)'],
    'Akurasi Keberhasilan': [f"{akurasi_dm * 100:.1f}%", f"{akurasi_ml * 100:.1f}%"]
})

print("=== HASIL AKHIR PENGGALIAN SUMUR ===")
print(df_komparasi.to_string(index=False))

=== HASIL AKHIR PENGGALIAN SUMUR ===
                      Tim Akurasi Keberhasilan
   Tim DM (Intuisi/Rules)                73.0%
Tim ML (Predictive Model)                96.0%
